# Decision Trees

Decision trees are the anti-linear model. Where linear regression assumes a straight-line relationship between features and target, trees make no assumptions at all. They partition the feature space into rectangular regions using a sequence of if-then rules, learning the structure directly from the data.

Key properties that set trees apart:

- No scaling required - trees split on thresholds, so they're invariant to monotonic transformations of features. Remember our "when to scale" framework from the Pipeline lecture: distance-based methods (KNN) and regularization (Ridge, Lasso) need scaling; trees do not.
- Handles feature interactions - a split on feature A followed by a split on feature B captures their interaction without you specifying it. Contrast this with regression, where we had to manually create interaction terms with `PolynomialFeatures`.
- Automatic feature elimination - the splitting process handles this automatically by selecting the features that contribute the most
- Interpretable rules - you can trace any prediction from root to leaf and explain *why*

In bias-variance terms, decision trees are *low bias, high variance*. They're flexible enough to fit complex patterns (low bias) but sensitive to the specific training data they see (high variance). A small change in the training set can produce a completely different tree. This is the opposite of linear regression, which is high bias (rigid) but low variance (stable). Pruning and ensemble methods (next lecture) address the variance problem.

Changes after Day 21 lecture (2026-03-26):

- [x] Add clear bias-variance framing: trees are low bias, high variance (opposite of linear models)
- [x] Clarify that CART produces binary *splits*, not limited to binary classification
- [x] Note that sklearn builds trees depth-first (not breadth-first)
- [x] Add note about oblique trees for diagonal boundaries (not in sklearn)

## Setup

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.metrics import accuracy_score, classification_report, mean_squared_error, r2_score
from sklearn.model_selection import (
    cross_val_score,
    GridSearchCV,
    train_test_split,
    validation_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

---

## Part 1: Classification Trees

We'll start with Iris - familiar from the SVM lecture - using all four features.

In [ ]:
iris = load_iris()
X_cls, y_cls = iris.data, iris.target

print(iris.DESCR)

sklearn's `DecisionTreeClassifier` follows the same `fit`/`predict`/`score` interface as every other model we've used. Under the hood, it uses the CART algorithm (Classification and Regression Trees), which always produces *binary splits* - each node divides into exactly two children. CART handles any number of classes; it's the splits that are binary, not the classification task. The tree is built depth-first: it picks the best split at the root, then recursively splits the left child all the way down before moving to the right child. The key parameter is `max_depth`, which limits how many levels of splits the tree can make. The slides covered *how* the tree decides where to split (Gini impurity, information gain). Here we focus on using the model and interpreting its output.

In [ ]:
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.3, random_state=42
)

tree_clf = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_clf.fit(X_train_cls, y_train_cls)

y_pred_cls = tree_clf.predict(X_test_cls)
print(f"Test accuracy: {accuracy_score(y_test_cls, y_pred_cls):.2f}")

100% test accuracy! Perfect predictions with only 4 splits. Impressive. But this is only for a single train/test split. We might have gotten lucky with which observations ended up in the test set. Cross-validation gives a more honest estimate by evaluating across multiple splits.

In [ ]:
cv_scores = cross_val_score(tree_clf, X_cls, y_cls, cv=5)
print(f"CV scores: {cv_scores}")
print(f"Mean: {cv_scores.mean():.2f} (+/- {cv_scores.std():.2f})")

It holds up very well, stable with almost perfect predictions. Iris is a relatively easy dataset - the species are well-separated in feature space. The decision tree achieves comparable performance to those models with no scaling, no Pipeline, and no assumptions about the shape of the decision boundary.

The more interesting question is *how* the tree achieves this, which we can answer by looking at the tree itself.

### Reading the tree

`plot_tree` renders the full tree structure as a figure. Each node shows the split rule, impurity, sample count, class distribution, and majority class. The `filled=True` option colors nodes by their majority class, with darker colors indicating higher purity.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
plot_tree(
    tree_clf,
    filled=True,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    rounded=True,
    fontsize=11,
    ax=ax,
)
ax.set_title("Decision Tree for Iris Classification")
plt.tight_layout()
plt.show()

print(f"Nodes: {tree_clf.tree_.node_count} | Depth: {tree_clf.get_depth()} | Leaves: {tree_clf.get_n_leaves()}")

Each node shows:

- The split rule (feature and threshold)
- Gini impurity (0.0 = pure, 0.5 = maximum uncertainty for binary)
- Number of samples reaching this node
- Class distribution and majority class

This is the interpretability advantage of decision trees - you can explain any prediction by tracing the path from root to leaf. Let's walk through a few nodes:

The *root node* (top) asks: is petal length <= 2.45 cm? All 105 training samples enter here, with a Gini impurity of 0.664 (high - all three classes are present). This single question perfectly separates setosa: all 31 setosa samples go left, arriving at a pure leaf (gini = 0.0). One rule, one feature, and an entire species is identified.

The *right branch* (petal length > 2.45) contains the remaining 74 samples - all versicolor and virginica. Gini is 0.500, meaning the two classes are evenly mixed. The next split, again on petal length (<= 4.75), creates a nearly pure versicolor group on the left (33 samples, gini = 0.059 - only 1 is misclassified) and a mostly-virginica group on the right (41 samples, gini = 0.214).

The deeper splits at levels 3-4 clean up boundary cases between versicolor and virginica using petal width as a secondary feature. These deeper nodes have fewer samples and higher residual impurity - they're working harder for smaller gains.

Key observations:

- The tree chose petal features for every split - sepal measurements aren't useful here. This is the tree automatically performing feature selection through its splitting process.
- Setosa is trivially separable. The tree's real work is distinguishing versicolor from virginica in the overlap region.
- Each level of depth handles progressively finer distinctions. The first split does the most work; later splits are cleanup.

### Decision boundaries

Trees split on one feature at a time, so their decision boundaries are always axis-aligned rectangles. Using just two features makes this visible. We'll use `DecisionBoundaryDisplay`, the same tool we used on Day 20 for SVM - it works with any classifier.

In [ ]:
# Fit on petal features only (for visualization)
X_2d = iris.data[:, 2:4]  # petal length, petal width
tree_2d = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_2d.fit(X_2d, iris.target)

fig, axes = plt.subplots(figsize=(7, 4))

# Decision boundaries
DecisionBoundaryDisplay.from_estimator(
    tree_2d, X_2d, ax=axes,
    cmap="RdYlBu", alpha=0.3, response_method="predict",
)
scatter = axes.scatter(
    X_2d[:, 0], X_2d[:, 1], c=iris.target,
    cmap="RdYlBu", edgecolor="k", s=30,
)
axes.set_xlabel("Petal Length (cm)")
axes.set_ylabel("Petal Width (cm)")
axes.set_title("Decision Boundaries (rectangular splits)")

The rectangular boundaries are a signature of decision trees - every boundary is a horizontal or vertical line because each split acts on one feature at a time. Compare this with the SVM boundaries from Day 20: SVM can draw curved boundaries (kernel trick), trees cannot. Trees compensate by stacking more splits to approximate curved regions.

Let's look at the tree behind this plot to connect the boundaries to specific split decisions.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(
    tree_2d,
    filled=True,
    feature_names=["petal length (cm)", "petal width (cm)"],
    class_names=iris.target_names,
    rounded=True,
    fontsize=11,
    ax=ax,
)
ax.set_title("2-Feature Decision Tree (the tree behind the boundary plot)")
plt.tight_layout()
plt.show()

Each split in this tree corresponds to a line in the boundary plot. The root split (petal length <= 2.45) draws the vertical line that separates setosa (left) from everything else. On the right branch, petal width <= 1.75 draws a horizontal line, and petal length <= 4.95 draws another vertical line - together these two splits create the rectangular region where versicolor lives. Everything outside that box is classified as virginica. These axis-aligned rectangles are the only shapes a standard decision tree can produce - it can't draw the diagonal or curved boundaries that SVM or logistic regression might find. (Variants called *oblique trees* split on linear combinations of features and can produce diagonal boundaries, but these aren't implemented in sklearn's `DecisionTreeClassifier`.)

What can this tree tell us about feature importance?

In [ ]:
fig, axes = plt.subplots(figsize=(6, 3))

# Feature importance
importance = tree_2d.feature_importances_
axes.barh(["Petal Length", "Petal Width"], importance)
axes.set_xlabel("Importance (MDI)")
axes.set_title("Feature Importance")

plt.tight_layout()
plt.show()

print(f"2-feature accuracy: {tree_2d.score(X_2d, iris.target):.2f}")

The second plot shows *feature importance*, measured by mean decrease in impurity (MDI). Each time a feature is used for a split, it reduces the Gini impurity by some amount. MDI sums these reductions across all nodes in the tree, weighted by the number of samples at each node. Features that create purer splits higher in the tree get higher importance scores. We'll revisit importance in the next lecture when we compare MDI with a more robust alternative.

---

## Part 2: Regression Trees

The same algorithm works for regression with two changes. Instead of Gini impurity, splits minimize MSE. Instead of a class vote, each leaf predicts the mean of its training samples.

We'll use the California Housing dataset - 20,640 block groups from the 1990 Census, with 8 features (median income, house age, rooms, etc.) predicting median house value. This is a larger, messier dataset than Iris, and it's a regression problem - a good test of whether trees generalize beyond classification.

In [ ]:
california = fetch_california_housing()
X_reg, y_reg = california.data, california.target

print(california.DESCR)

In [ ]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.3, random_state=42
)

tree_reg = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_reg.fit(X_train_reg, y_train_reg)

y_pred_reg = tree_reg.predict(X_test_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
r2 = r2_score(y_test_reg, y_pred_reg)
print(f"RMSE: {rmse:.2f} | R²: {r2:.2f}")

An R² of ~0.52 with only 3 levels of splits. Compare this with the linear models from Unit 2 on similar data - the tree captures some non-linear structure that linear regression misses, but it's also constrained by the shallow depth.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 10))
plot_tree(
    tree_reg,
    filled=True,
    feature_names=california.feature_names,
    rounded=True,
    fontsize=10,
    ax=ax,
)
ax.set_title("Regression Tree for California Housing")
plt.tight_layout()
plt.show()

In a regression tree, the "value" in each node is the mean target value for all samples in that region. The color intensity reflects the predicted value - darker orange means higher house prices.

Walk through the tree from the root:

The *root node* splits on MedInc <= 5.03 (\\$50,300 median income). All 14,448 training samples enter here with a mean house value of \\$207k and MSE of 1.34. This one split separates lower-income areas (left, 11,340 samples, mean \\$173k) from higher-income areas (right, 3,108 samples, mean \\$331k).

The *left branch* (lower income) splits again on MedInc <= 3.07. Below \\$30,700, the tree looks at AveRooms - areas with fewer rooms per household (< 4.2) have slightly higher values (\\$166k vs \\$118k), likely reflecting dense urban housing. Above \\$30,700 income, the split is on AveOccup - less crowded households (< 2.35 people) predict higher values (\\$282k vs \\$188k).

The *right branch* (higher income) splits once more on MedInc <= 6.87. The wealthiest areas (MedInc > 8.16) predict the highest values at \\$466k, while the MedInc 5-7 range shows occupancy effects similar to the left branch.

Notice that MedInc dominates the top of the tree - it appears in 4 of the 7 split nodes. The tree has discovered the same income-price relationship that Ridge regression captures linearly, but it's expressing it as threshold rules that can capture non-linear jumps (the difference between \\$50k and \\$70k income matters more than the difference between \\$30k and \\$50k).

### Feature importance

In [ ]:
importance_reg = tree_reg.feature_importances_
sorted_idx = np.argsort(importance_reg)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(np.array(california.feature_names)[sorted_idx], importance_reg[sorted_idx])
ax.set_xlabel("Importance (MDI)")
ax.set_title("Feature Importance - California Housing")
plt.tight_layout()
plt.show()

`MedInc` dominates because it appears in the earliest splits, where the most samples flow through. With only `max_depth=3`, the tree can use at most 7 split points total, so only a few features get a chance to contribute. Deeper trees distribute importance more broadly.

### Step-function predictions

Regression trees approximate continuous relationships with piecewise constant functions - flat segments separated by sharp jumps at split thresholds. This is easiest to see with one feature.

In [ ]:
most_important = np.argmax(importance_reg)
feat_name = california.feature_names[most_important]

# Single-feature tree for visualization
X_single = X_train_reg[:, [most_important]]
tree_single = DecisionTreeRegressor(max_depth=3, random_state=42)
tree_single.fit(X_single, y_train_reg)

X_range = np.linspace(
    X_reg[:, most_important].min(),
    X_reg[:, most_important].max(),
    1000,
).reshape(-1, 1)
y_range = tree_single.predict(X_range)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(X_train_reg[:, most_important], y_train_reg, alpha=0.1, s=5, label="Training data")
ax.step(X_range.flatten(), y_range, color="red", linewidth=2, where="post", label="Tree predictions")
ax.set_xlabel(feat_name)
ax.set_ylabel("Median House Value ($100k)")
ax.set_title(f"Piecewise Constant Approximation (max_depth=3)")
ax.legend()
plt.tight_layout()
plt.show()

Each flat segment is one leaf node. With `max_depth=3`, the tree can make, at most, $2^3 = 8$ regions. The red step function captures the general upward trend between income and house price, but it can't capture the curvature within each step. More depth means more steps and a closer approximation - but also more overfitting risk. This tension between approximation quality and overfitting is the central challenge of decision trees.

---

## Part 3: Overfitting and Tuning

Trees will keep splitting until every leaf is pure (classification) or contains one sample (regression). An unconstrained tree memorizes the training data. We can see this by comparing a depth-3 tree with a tree that has no depth limit:

In [ ]:
# Unconstrained tree vs shallow tree
tree_deep = DecisionTreeRegressor(random_state=42)  # no max_depth
tree_deep.fit(X_train_reg, y_train_reg)

train_rmse_deep = np.sqrt(mean_squared_error(y_train_reg, tree_deep.predict(X_train_reg)))
test_rmse_deep = np.sqrt(mean_squared_error(y_test_reg, tree_deep.predict(X_test_reg)))
train_rmse_shallow = np.sqrt(mean_squared_error(y_train_reg, tree_reg.predict(X_train_reg)))
test_rmse_shallow = np.sqrt(mean_squared_error(y_test_reg, tree_reg.predict(X_test_reg)))

print("Unconstrained tree:")
print(f"  Train RMSE: {train_rmse_deep:.4f} | Test RMSE: {test_rmse_deep:.4f}")
print(f"  Depth: {tree_deep.get_depth()} | Leaves: {tree_deep.get_n_leaves()}")
print()
print("Shallow tree (max_depth=3):")
print(f"  Train RMSE: {train_rmse_shallow:.4f} | Test RMSE: {test_rmse_shallow:.4f}")
print(f"  Depth: {tree_reg.get_depth()} | Leaves: {tree_reg.get_n_leaves()}")

Remember, we're working with RMSE here, so lower is better, and test > train is expected.

The unconstrained tree achieves near-zero training error (it has memorized every training example in its own leaf node) but its test error is *worse* than the shallow tree's. That gap between training and test performance is the textbook definition of overfitting. The unconstrained tree has nearly 14,000 leaves - far more than it needs to capture the real patterns in the data.

### Validation curve: how depth affects performance

We saw `validation_curve` in the KNN lecture (07a) where we swept over k. The same tool works for any hyperparameter. Here we sweep over `max_depth` to see where the bias-variance tradeoff lives for this dataset.

In [ ]:
depths = np.arange(1, 21)
train_scores, test_scores = validation_curve(
    DecisionTreeRegressor(random_state=42),
    X_train_reg, y_train_reg,
    param_name="max_depth",
    param_range=depths,
    cv=5,
    scoring="neg_root_mean_squared_error",
)

# validation_curve returns negative RMSE (sklearn convention: higher = better)
train_rmse_vc = -train_scores.mean(axis=1)
test_rmse_vc = -test_scores.mean(axis=1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(depths, train_rmse_vc, "o-", label="Train RMSE")
ax.plot(depths, test_rmse_vc, "s-", label="CV RMSE")
ax.set_xlabel("max_depth")
ax.set_ylabel("RMSE")
ax.set_title("Validation Curve: Depth vs. Performance")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_depth = depths[np.argmin(test_rmse_vc)]
print(f"Best depth by CV: {best_depth}")

The training error keeps dropping as the tree memorizes more detail, but the CV error bottoms out before beginning a slow rise. The gap between the two curves is overfitting - the same pattern we saw with polynomial degree in regression and with k in KNN (where small k = more complex, the opposite direction). The best depth balances the two.

### GridSearchCV over tree hyperparameters

`max_depth` isn't the only knob. Tree complexity is also controlled by:

- `min_samples_split` - a node must have at least this many samples to be split further
- `min_samples_leaf` - every leaf must contain at least this many samples

All three limit growth and reduce overfitting. `GridSearchCV` can search over multiple parameters simultaneously, just as we did for KNN and Ridge.

In [ ]:
pipe = Pipeline([
    ("scaler", StandardScaler()),  # trees don't need this - included for pipeline consistency
    ("tree", DecisionTreeRegressor(random_state=42)),
])

param_grid = {
    "tree__max_depth": [3, 5, 7, 10, 15],
    "tree__min_samples_leaf": [1, 5, 10, 20],
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="neg_root_mean_squared_error")
grid.fit(X_train_reg, y_train_reg)

print(f"Best params: {grid.best_params_}")
print(f"Best CV RMSE: {-grid.best_score_:.4f}")
print(f"Test RMSE:    {np.sqrt(mean_squared_error(y_test_reg, grid.predict(X_test_reg))):.4f}")

Note that `StandardScaler` is unnecessary for decision trees since they split on thresholds, not distances. It's included here to show the familiar Pipeline pattern and because it does no harm - the results are identical with or without it. Try removing the scaler to verify.

The tuned tree substantially outperforms both the depth-3 tree and the unconstrained tree. The optimal combination typically uses moderate depth with a `min_samples_leaf` constraint that prevents the tree from creating tiny, overfit leaf nodes.

### A note on pruning

An alternative to pre-pruning (constraining depth, min samples) is post-pruning: grow a full tree, then cut back branches that don't improve generalization. sklearn implements this via `ccp_alpha` (cost-complexity pruning). Larger alpha values produce simpler trees.

The idea is the same as regularization in linear models: add a penalty for complexity to prevent overfitting. sklearn's `cost_complexity_pruning_path` computes the sequence of alpha values where the tree structure changes, and we can evaluate each one.

In [ ]:
# Grow a full tree, then get the pruning path
full_tree = DecisionTreeRegressor(random_state=42)
path = full_tree.cost_complexity_pruning_path(X_train_reg, y_train_reg)
ccp_alphas = path.ccp_alphas

# Sample alpha values log-spaced across the range (there are thousands)
# Log-spacing gives better coverage across orders of magnitude
alpha_min = ccp_alphas[ccp_alphas > 0].min()
alpha_max = ccp_alphas[-2]  # skip the trivial last one
sampled_alphas = np.logspace(np.log10(alpha_min), np.log10(alpha_max), 20)

# Fit a tree at each alpha and record train/test RMSE
train_rmses, test_rmses, n_leaves = [], [], []
for alpha in sampled_alphas:
    t = DecisionTreeRegressor(ccp_alpha=alpha, random_state=42)
    t.fit(X_train_reg, y_train_reg)
    train_rmses.append(np.sqrt(mean_squared_error(y_train_reg, t.predict(X_train_reg))))
    test_rmses.append(np.sqrt(mean_squared_error(y_test_reg, t.predict(X_test_reg))))
    n_leaves.append(t.get_n_leaves())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Left: RMSE (zoomed to show the improvement)
ax1.plot(sampled_alphas, train_rmses, "o-", label="Train RMSE")
ax1.plot(sampled_alphas, test_rmses, "s-", label="Test RMSE")
ax1.set_xlabel("ccp_alpha")
ax1.set_ylabel("RMSE")
ax1.set_xscale("log")
ax1.set_ylim(0, 1.05)
ax1.set_title("RMSE vs. Pruning Strength")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: Tree complexity
ax2.plot(sampled_alphas, n_leaves, "^-", color="gray")
ax2.set_xlabel("ccp_alpha")
ax2.set_ylabel("Number of leaves")
ax2.set_xscale("log")
ax2.set_title("Tree Complexity vs. Pruning Strength")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

best_idx = np.argmin(test_rmses)
print(f"Best alpha: {sampled_alphas[best_idx]:.6f} | Leaves: {n_leaves[best_idx]} | Test RMSE: {test_rmses[best_idx]:.4f}")

As alpha increases (moving right), the tree shrinks from ~14,000 leaves to just a few. Training RMSE rises steadily (the tree can memorize less), but test RMSE *improves* in the middle range before worsening at the far right where the tree is too simple. The sweet spot is a tree with under 100 leaves - drastically smaller than the unconstrained 14,000 but with better generalization.

Both pre-pruning (GridSearchCV over `max_depth` and `min_samples_leaf`) and post-pruning (`ccp_alpha`) address the same problem from different directions. Pre-pruning is more common in practice because it's faster - you never grow the branches you'd later cut. But post-pruning can find tree structures that no combination of pre-pruning constraints would produce.

### Trees overfit - ensembles fix this

Single trees face a fundamental tension: shallow trees underfit (too few regions to capture the data's structure), deep trees overfit (too many regions, fitting noise). No single depth is ideal for all parts of the feature space.

What if instead of one tree, we grew many trees on different samples of the data and averaged their predictions? The individual trees would still overfit, but their errors would be different - and averaging would cancel out the noise while preserving the signal. That's the idea behind *ensemble methods*, and it's where we're headed next.

## Summary

| Property | Decision Trees |
|---|---|
| Task | Classification and regression |
| Scaling required | No - splits on thresholds |
| Handles interactions | Yes - automatic via nested splits |
| Interpretability | High - trace root to leaf |
| Overfitting risk | High - unconstrained trees memorize |
| Key hyperparameters | `max_depth`, `min_samples_split`, `min_samples_leaf` |

New APIs introduced today:

| API | Purpose |
|---|---|
| `DecisionTreeClassifier` | Classification tree |
| `DecisionTreeRegressor` | Regression tree |
| `plot_tree` | Visualize tree structure |
| `feature_importances_` | Mean decrease in impurity (MDI) |
| `validation_curve` | Sweep a hyperparameter with CV |